In [3]:
import h5py 
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC, LinearSVC
from sklearn import svm  # Added for svm.LinearSVC reference
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ========== 1. SPLIT DATA ==========
def split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)


# ========== 2. HYPERPARAMETER TUNING ==========
def cross_validate_svm(X_train, y_train, cv=5):
    """
    Perform cross-validation for SVM hyperparameters (C and kernel).
    """
    param_grid = {
        'estimator__C': [0.1, 1, 10],
    }
    
    # dual=True when n_samples < n_features 
    ovr = OneVsRestClassifier(svm.LinearSVC(max_iter=1000, dual=True,
                                            random_state=0))

    grid_search = GridSearchCV(ovr, param_grid, cv=cv, n_jobs=-1, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Cross-validate best model to compute average accuracy
    best_model = grid_search.best_estimator_
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    mean_accuracy = np.mean(cv_scores)
    std_accuracy = np.std(cv_scores)

    return best_model, grid_search.best_params_, mean_accuracy, std_accuracy

# ========== 3. MAIN FUNCTION ==========
def run_one_vs_rest_svm(X, y):
    # Step 1: Train-test split
    X_train, X_test, y_train, y_test = split_data(X, y)

    # Step 2: Cross-validation
    print("Running cross-validation and hyperparameter tuning...")
    best_model, best_params, mean_acc, std_acc = cross_validate_svm(X_train, y_train)

    print(f"\nBest Parameters: {best_params}")
    print(f"Cross-Validation Mean Accuracy: {mean_acc:.4f}")
    print(f"Cross-Validation Std Dev: {std_acc:.4f}")

    # Step 3: Train best model on full training set and evaluate
    print("\nEvaluating on test set...")
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    print(f"Test Accuracy: {acc:.4f}")
    print("Classification Report:\n", report)

    return {
        'model': best_model,
        'cv_mean_accuracy': mean_acc,
        'cv_std_accuracy': std_acc,
        'test_accuracy': acc,
        'classification_report': report,
        'best_params': best_params
    }

In [4]:
act_path = "/om/scratch/Fri/imgriff/binaural_unit_activations_for_SVM/word_task_v10_main_feature_gain_config/word_task_v10_main_feature_gain_config_model_activations_for_word_SVM_val.h5"
h5_file = h5py.File(act_path, 'r', swmr=True)

In [3]:
from pprint import pprint
pprint(list(h5_file.keys()))

['cochleagram',
 'conv_block_0_relu',
 'conv_block_1_relu',
 'conv_block_2_relu',
 'conv_block_3_relu',
 'conv_block_4_relu',
 'conv_block_5_relu',
 'conv_block_6_relu',
 'layer_names',
 'relufc',
 'target_word_int']


In [13]:
layer_name = 'conv_block_0_relu'

acts = h5_file[layer_name]
labels = h5_file['target_word_int'][:]
# compute memory footprint of acts 
print(acts.nbytes / (1024 ** 3), "GB")


464.1491174697876 GB


In [14]:
svm_results = run_one_vs_rest_svm(acts, labels)


Running cross-validation and hyperparameter tuning...


KeyboardInterrupt: 